# Solutions - Column Operations (Easy 14)


In [ ]:
from pyspark.sql import functions as F, types as T

transactions = spark.table("samples.bakehouse.sales_transactions")
orders = spark.table("samples.tpch.orders")
bookings = spark.table("samples.wanderbricks.bookings")


## Problem 1 - Column Casting

Cast `cardNumber` to string and `totalPrice` to double.


In [ ]:
result_1 = transactions.select(
    "transactionID", "cardNumber",
    F.col("cardNumber").cast(T.StringType()).alias("card_str"),
    "totalPrice",
    F.col("totalPrice").cast(T.DoubleType()).alias("revenue_double"),
    "paymentMethod"
)
result_1.display()


## Problem 2 - Rename and Drop

Rename key columns and drop unnecessary ones from the orders table.


In [ ]:
result_2 = orders.withColumnRenamed(
    "o_orderkey", "order_id"
).withColumnRenamed(
    "o_custkey", "customer_id"
).withColumnRenamed(
    "o_totalprice", "total_price"
).drop("o_comment", "o_clerk", "o_shippriority")
result_2.display()

## Problem 3 - Price Category

Categorise transactions into Low/Medium/High price buckets and aggregate.


In [ ]:
result_3 = transactions.withColumn("price_category",
    F.when(F.col("totalPrice") < 20, "Low")
     .when(F.col("totalPrice") <= 100, "Medium")
     .otherwise("High")
).groupBy("price_category").agg(
    F.count("*").alias("transaction_count"),
    F.round(F.avg("totalPrice"), 2).alias("avg_price")
)
result_3.display()


## Problem 4 - Literal and Expression

Add a platform literal, calculate stay nights, and derive per-night price.


In [ ]:
result_4 = bookings.select(
    "booking_id",
    F.lit("Wanderbricks").alias("platform"),
    "check_in", "check_out",
    F.expr("datediff(check_out, check_in)").alias("stay_nights"),
    "total_amount",
    F.round(F.col("total_amount") / F.expr("datediff(check_out, check_in)"), 2).alias("per_night_price")
)
result_4.display()


## Problem 5 - Select with Expressions

Select and rename transaction columns using expressions.


In [ ]:
result_5 = transactions.select(
    F.col("transactionID").alias("transaction_id"),
    F.to_date("dateTime").alias("transaction_date"),
    "product",
    F.col("quantity").alias("qty"),
    F.col("unitPrice").alias("unit_price"),
    F.col("totalPrice").alias("total_price"),
    F.col("paymentMethod").alias("payment_type")
)
result_5.display()
